# SDS210 Final project - Wildfire Mapping in Borneo

This notebook builds an automated pipeline that fetches live satellite data from NASA FIRMS, cleans it and visualizes it on an interactive map.

**Research question:** Where are the most intense active fires in Borneo right now, and how is fire activity distributed across the region?

## 1. Imports

In [14]:
import pandas as pd
import geopandas as gpd
import requests
import folium
from folium.plugins import HeatMap
import matplotlib.pyplot as plt
from io import StringIO

## 2. API Configuration

- NASA FIRMS created a free API for the near-real-time fire data (sent via emaail).
- The personal map key is needed and can be obtained here 
--> https://firms.modaps.eosdis.nasa.gov/api/
- the key goes into every data request

In [ ]:
#firms API

#still do in git ignore later

map_key   = "your_key_here"

#get url, hide map key 
url_api = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?map_key=' + map_key

#check ststus and how many transactions i have

response = requests.get(url_api)
print(response.status_code)
print(response.text[:200])


200
{ "transaction_limit" : 5000, "current_transactions": 0, "transaction_interval" : "10 minutes" }


## 3. Data aviability
- Quesry to see which sensors and date ranges are aviable through the API. 
- url will return info about all supported sensors and their corresponding datasets
- instead of 'all' i could also specify individual sensor like landsat_nrt
- The URL structure is always: 'base_url' + map_key + '/dataset_I_want'

In [16]:
##base_url / my_key / what_I_want
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + map_key + '/all'
df = pd.read_csv(da_url)
display(df)

,data_id,min_date,max_date
0,MODIS_NRT,2026-03-01,2026-05-19
1,MODIS_SP,2000-11-01,2026-02-28
2,VIIRS_NOAA20_NRT,2026-04-01,2026-05-18
3,VIIRS_NOAA20_SP,2018-04-01,2026-03-31
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-19
5,VIIRS_SNPP_NRT,2026-04-01,2026-05-19
6,VIIRS_SNPP_SP,2012-01-20,2026-03-31
7,LANDSAT_NRT,2022-06-20,2026-05-18
8,GOES_NRT,2022-08-09,2026-05-19
9,BA_MODIS,2000-11-01,2026-02-01


## 4. Parameters

- The three VIIRS sensors are being chose, max coverage, near_real_time data
- define bouding box (borneo)
- time window


In [22]:
## Borneo BBOX: west, south, east, north

bbox = "108.5,-4.5,119.5,7.5"

# Sensoren: all three VIIRS NRT --> near real time
sensors = ["VIIRS_SNPP_NRT", "VIIRS_NOAA20_NRT", "VIIRS_NOAA21_NRT"]

# how many days i wanna look at
day_range = 2

## 5. Download and clean the fire data
The 'fetch()' function  builds the api_url for sensors and return dataframe. Calles once per sonsors, then all the results are stacked into one dataset

Quality filter remove low confidence detection 'l' and dead pixels with no measured 'frp=0'


In [23]:
def fetch(sensor):
    """Downloads FIRMS fire data for one sensor as a DataFrame."""
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/{sensor}/{bbox}/{day_range}"
    r = requests.get(url, timeout=60)
    r.raise_for_status() #raises error if request failed
    df = pd.read_csv(StringIO(r.text))
    df["sensor"] = sensor #records which satellite detected this fire
    return df

# Download & combine all sensors and combine into one DataFrame
df = pd.concat([fetch(s) for s in sensors], ignore_index=True)

# Keep only nominal/high confidence detections with real fire energy
df = df[df["confidence"].isin(["n", "h"]) & (df["frp"] > 0)]

print(f"{len(df)} fire detections after cleaning")
print(df.groupby("sensor").size())
df.head()


71 fire detections after cleaning
sensor
VIIRS_NOAA20_NRT    12
VIIRS_NOAA21_NRT    26
VIIRS_SNPP_NRT      33
dtype: int64


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,sensor
0,-1.89548,115.86476,330.90,0.39,0.36,2026-05-18,550,N,VIIRS,n,2.0NRT,282.59,3.48,D,VIIRS_SNPP_NRT
1,3.88193,116.88798,333.24,0.41,0.37,2026-05-18,550,N,VIIRS,n,2.0NRT,290.73,2.80,D,VIIRS_SNPP_NRT
2,3.96824,117.25002,335.54,0.42,0.38,2026-05-18,550,N,VIIRS,n,2.0NRT,287.99,3.69,D,VIIRS_SNPP_NRT
3,5.21342,114.74483,334.19,0.38,0.36,2026-05-18,553,N,VIIRS,n,2.0NRT,286.71,2.41,D,VIIRS_SNPP_NRT
4,5.82632,114.34238,342.97,0.56,0.51,2026-05-18,1828,N,VIIRS,n,2.0NRT,274.58,8.22,N,VIIRS_SNPP_NRT


## 6. Create interactive map

Two layers combined to one folium map

1.  **Heatmap** --> all fire detections weighted by FRP. Shows the overall spatial pattern of fire activity across Borneo.

2. **Hot fire markers** — individual circle markers for the top 10% most intense fires (by FRP). By clicking the marker, one can see DRP vale, date and sensor. --> most energetically significant fires.

you can switch between the layer with the layer control 

In [26]:

# build interctive map with folium
m = folium.Map(location=[1.5, 114], zoom_start=6, tiles="CartoDB positron")

# Layer 1: Heatmap layer (weighted by FRP)
heat_data = df[["latitude", "longitude", "frp"]].values.tolist()
HeatMap(heat_data, radius=8, blur=6, name="Fire intensity (FRP)").add_to(m)

# layer 2:dot markers for high-FRP fires (top 10%)
threshold = df["frp"].quantile(0.90)
hot = df[df["frp"] >= threshold]

dots = folium.FeatureGroup(name=f"Hot fires (FRP ≥ {threshold:.0f} MW)")
for _, row in hot.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color="red",
        fill=True,
        fill_opacity=0.7,
        popup=f"FRP: {row['frp']:.1f} MW | {row['acq_date']} | {row['sensor']}",
    ).add_to(dots)
dots.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# Save
m.save("../outputs/borneo_fires.html")
print("Saved → borneo_fires.html")
m

Saved → borneo_fires.html


## 7. Summary

This pipeline automatically downloads, cleans, and maps near-real-time  wildfire data for Borneo from three VIIRS satellite sensors.

What are the findings: 
- Heatmap shows regional fire hotspots
- top 10% most intense fires (by FRP) are mapped individually as high risk
-Popup informations allow to get more information on intense fire events